In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from torchvision import transforms

#IMG_SIZE = (256, 512)  # (H, W); start small
#IMG_SIZE = (128, 256)  # (H, W); start small
IMG_SIZE = (64, 128)  # (H, W); start small

# Images are continuous colour gradients, while the masks are categorical labels. Use different interpolation methods for each.
# Docs: https://docs.pytorch.org/vision/0.9/_modules/torchvision/transforms/functional.html#InterpolationMode
image_transform = transforms.Compose([
    # BILINEAR averages neighboring pixels to produce smooth gradients
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor() # tensor scaled to [0, 1]
])

target_transform = transforms.Compose([
    # NEAREST preserves original values
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(), # tensor without scaling (stays 0-255)
    # remove dummy channel dimension and cast to int64 because PyTorch's CrossEntropyLoss requires this for segmentation. [1, H, W] -> [H, W]
    transforms.Lambda(lambda x: x.squeeze(0).long()),
])

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.dataset import CityscapesDataset

train_ds = CityscapesDataset("./data/dataset/train", image_transform, target_transform)
valid_ds = CityscapesDataset("./data/dataset/valid", image_transform, target_transform)
test_ds = CityscapesDataset("./data/dataset/test", image_transform, target_transform)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=4, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=True, num_workers=0)

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=19, base=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.d1 = DoubleConv(in_channels, base)
        self.d2 = DoubleConv(base, base*2)
        self.d3 = DoubleConv(base*2, base*4)
        self.d4 = DoubleConv(base*4, base*8)
        self.bottleneck = DoubleConv(base*8, base*16)

        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.c4 = DoubleConv(base*16, base*8)

        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.c3 = DoubleConv(base*8, base*4)

        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.c2 = DoubleConv(base*4, base*2)

        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.c1 = DoubleConv(base*2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        c4 = self.d4(self.pool(c3))
        b  = self.bottleneck(self.pool(c4))

        x = self.c4(torch.cat([self.u4(b), c4], dim=1))
        x = self.c3(torch.cat([self.u3(x), c3], dim=1))
        x = self.c2(torch.cat([self.u2(x), c2], dim=1))
        x = self.c1(torch.cat([self.u1(x), c1], dim=1))
        return self.out(x)

In [ ]:
model = UNet().to(device)

imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # [B,H,W]   int64
print("mask min/max:", masks.min().item(), masks.max().item())  # 0..18 and/or 255

with torch.no_grad():
    out = model(imgs.to(next(model.parameters()).device))
print("logits:", out.shape)  # [B,19,H,W]

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

model = UNet(in_ch=3, num_classes=19, base=32).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks in tqdm(loader, leave=False):
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).long()

            logits = model(imgs)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 20
best_val = float("inf")

for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, True)
    va = run_epoch(valid_loader, False)
    print(f"epoch {epoch:02d}  train {tr:.4f}  val {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), "./models/best_unet.pt")

In [ ]:
model.load_state_dict(torch.load("./models/best_unet.pt"))
model.eval()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

imgs, masks = next(iter(test_loader))
with torch.no_grad():
    preds = model(imgs.to(device)).argmax(1).cpu()

fig, ax = plt.subplots(3, 1, figsize=(8, 10))
ax[0].imshow(imgs[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)  # rough denorm
ax[1].imshow(masks[0].numpy(), vmin=0, vmax=18)
ax[2].imshow(preds[0].numpy(), vmin=0, vmax=18)
plt.show()